# Grid modules from autocorrelogram structure — UMAP + clustering

Takes the sessions with the highest grid-cell yield from
`create_classification_datasheet_v2.ipynb` and asks whether the cells organise
into **discrete clusters by autocorrelogram shape**, and whether those clusters
look like grid modules (distinct field spacings) or something adjacent —
"grid-associated" structure that is periodic but not cleanly hexagonal.

**Why the autocorrelogram rather than the rate map.** The spatial
autocorrelogram (SAC) is translation-invariant: two grid cells of the same
module with different spatial phases have near-identical SACs but completely
different rate maps. Clustering on SACs therefore groups by *periodic structure*
— spacing and orientation — which is what a module is, rather than by where the
fields happen to sit.

**Method** follows Gardner et al. 2022 (Extended Data Fig. 2): UMAP with
`metric='manhattan'`, `n_neighbors=5`, `min_dist=0.05`, `init='spectral'`;
DBSCAN with a minimum of 30 points per cluster and eps swept over 0.6-1.0.

Scores and classifications come from the v2 datasheet cache, so **run that
notebook's sweep first** (`RUN_FULL_SWEEP = True`) — this one reads what it left
behind.

In [1]:
import sys, os, glob, time, warnings
sys.path.insert(0, '/Users/harryclark/Documents/spatial-manifolds/src')
sys.path.insert(0, '/Users/harryclark/Documents/spatial-manifolds/GRID-PATTERN-FORMATION')
import numpy as np
import pandas as pd
import pynapple as nap
import matplotlib.pyplot as plt
from scipy.signal import fftconvolve
from scipy import ndimage
from skimage.feature import peak_local_max
from sklearn.cluster import DBSCAN
import umap
warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Arial'

SOURCE_PATH = '/Users/harryclark/Downloads/clark2025/'
CACHE_DIR   = '/Users/harryclark/Documents/spatial-manifolds/data/classification_v2_cache'
SAC_CACHE   = '/Users/harryclark/Documents/spatial-manifolds/data/sac_umap_cache'
os.makedirs(SAC_CACHE, exist_ok=True)

NB, BOUNDS, SMOOTH_SIGMA = 40, (0.0, 100.0, 0.0, 100.0), 1.5
ARENA_CM = 100.0
BIN_CM   = ARENA_CM / NB          # 2.5 cm per bin

MIN_GC_PER_SESSION = 15           # a session needs this many grid cells to be worth clustering
SAC_RADIUS_BINS    = NB // 2      # keep the central disc of the SAC; edges are unreliable

# UMAP / DBSCAN — Gardner et al. 2022 parameters
UMAP_KW  = dict(metric='manhattan', n_neighbors=5, min_dist=0.05,
                init='spectral', n_components=2, random_state=42)
DBSCAN_EPS, DBSCAN_MIN = 0.6, 30

def session_files(mouse, day, session):
    d = f'{SOURCE_PATH}M{int(mouse)}/D{int(day)}/{session}/'
    stem = f'sub-M{int(mouse)}_ses-D{int(day)}_typ-{session}'
    return d + stem + '_beh.nwb', d + stem + '_srt-kilosort4_clusters.npz'

# ── load the v2 classification datasheet ─────────────────────────────────────
# Read the assembled CSV rather than the per-session parquet cache: the CSV is the
# canonical output, it already carries anatomy, and it is stable while a shuffle
# top-up is rewriting the parquets underneath.
CLS_CSV = '/Users/harryclark/Documents/spatial-manifolds/data/cell_classifications_v2.csv'
_wide = pd.read_csv(CLS_CSV)
print(f'{len(_wide)} cells in {CLS_CSV}')

# CSV is wide (…_of1 / …_of2); melt back to one row per cell-session
_long = []
for _s in ('of1', 'of2'):
    _cols = {f'cell_class_{_s}': 'cell_class', f'grid_score_{_s}': 'grid_score',
             f'spatial_information_{_s}': 'spatial_information',
             f'hd_mean_vector_length_{_s}': 'hd_mean_vector_length',
             f'mean_rate_{_s}': 'mean_rate', f'peak_rate_{_s}': 'peak_rate'}
    _have = {k: v for k, v in _cols.items() if k in _wide.columns}
    _d = _wide[['mouse', 'day', 'cluster_id', 'gc_in', 'brain_region',
                'coord_SCs_x', 'coord_SCs_y', 'coord_CCFs_x'] + list(_have)].rename(columns=_have)
    _d['session'] = _s.upper()
    _long.append(_d)
scores = pd.concat(_long, ignore_index=True).dropna(subset=['cell_class'])

# ALL curated cells are embedded, not only grid cells: the question is how grid cells
# distribute within the full population's autocorrelogram structure, which cannot be
# asked if only grid cells are present.
scores = scores[~scores.cell_class.isin(['EXCL', 'LOW'])].copy()
print(f'{len(scores)} curated cell-sessions carried forward '
      f'{scores.cell_class.value_counts().to_dict()}')

gc_yield = (scores[scores.cell_class == 'GC'].groupby(['mouse', 'day', 'session'])
            .size().rename('n_gc').reset_index().sort_values('n_gc', ascending=False))
sel = gc_yield[gc_yield.n_gc >= MIN_GC_PER_SESSION]
print(f'\nsessions with >= {MIN_GC_PER_SESSION} grid cells: {len(sel)}')
print(sel.head(20).to_string(index=False) if len(sel) else '  (none)')


1 cached session-files, 308 cell-sessions

sessions with >= 15 grid cells:
 mouse  day session  n_gc
    25   24     OF1    50


## Autocorrelograms

Same `sac_fft` as the datasheet notebook (the definition lives there; repeated
here so this notebook runs standalone). Each cell's SAC is masked to its central
disc, flattened, and z-scored — UMAP on raw SACs would otherwise be dominated by
overall correlation magnitude rather than by periodic structure.

`sac_spacing` estimates grid spacing as the median distance from the SAC centre
to the six nearest surrounding peaks, converted to cm. For a hexagonal grid this
is the field spacing; for a non-grid cell it is whatever periodicity exists, so
it is only interpretable alongside the grid score.

In [2]:
def gaussian_filter_nan(arr, sigma):
    v = np.nan_to_num(arr, nan=0.0); w = np.isfinite(arr).astype(float)
    vs = ndimage.gaussian_filter(v, sigma=sigma, mode='nearest')
    ws = ndimage.gaussian_filter(w, sigma=sigma, mode='nearest')
    with np.errstate(invalid='ignore', divide='ignore'):
        out = vs / ws
    out[ws < 1e-8] = np.nan
    return out

def compute_rate_maps(tsgroup, beh):
    pos = np.stack([beh['P_x'], beh['P_y']], axis=1)
    tc, _ = nap.compute_2d_tuning_curves(tsgroup, pos, nb_bins=NB, minmax=BOUNDS,
                                          ep=beh['moving'])
    ids = list(tc.keys())
    maps = np.stack([gaussian_filter_nan(np.asarray(tc[i], float), SMOOTH_SIGMA)
                     for i in ids], axis=0)
    return ids, maps

def sac_fft(rmap, min_overlap=3):
    """2-D spatial autocorrelogram (FFT). Defined in the v2 datasheet notebook."""
    a = np.nan_to_num(np.asarray(rmap, float)); ones = np.ones_like(a)
    f = lambda b, x: fftconvolve(x, np.rot90(b, 2), mode='full')
    s1s2 = f(a, a); s1 = f(a, ones); s2 = f(ones, a)
    s1sq = f(a ** 2, ones); s2sq = f(ones, a ** 2)
    n = np.maximum(f(ones, ones), 1e-12); nsq = n ** 2
    v1 = np.maximum(s1sq / n - (s1 ** 2) / nsq, 0.0)
    v2 = np.maximum(s2sq / n - (s2 ** 2) / nsq, 0.0)
    denom = np.sqrt(v1 * v2); cov = s1s2 / n - (s1 * s2) / nsq
    out = np.zeros_like(cov)
    ok = (denom > 1e-12) & (np.round(f(ones, ones)) >= min_overlap)
    out[ok] = cov[ok] / denom[ok]
    return np.clip(out, -1.0, 1.0)

_c = NB - 1
_yy, _xx = np.mgrid[:2 * NB - 1, :2 * NB - 1]
SAC_DISC = ((_yy - _c) ** 2 + (_xx - _c) ** 2) <= SAC_RADIUS_BINS ** 2

def sac_vector(sac):
    """Central disc of the SAC, flattened and z-scored (shape, not magnitude)."""
    v = sac[SAC_DISC].astype(float)
    s = v.std()
    return (v - v.mean()) / s if s > 1e-9 else v - v.mean()

def sac_spacing(sac, min_distance=2):
    """Median centre-to-peak distance over the 6 nearest SAC peaks, in cm."""
    s = np.nan_to_num(sac)
    pk = peak_local_max(s, min_distance=min_distance, exclude_border=False)
    if len(pk) < 2:
        return np.nan
    d = np.hypot(pk[:, 0] - _c, pk[:, 1] - _c)
    d = np.sort(d[d > min_distance])
    return float(np.median(d[:6]) * BIN_CM) if len(d) else np.nan

def sac_peaks(sac, min_distance=2, n=6):
    """Offsets (dy, dx, dist_bins) of the n SAC peaks nearest the centre."""
    s = np.nan_to_num(sac)
    pk = peak_local_max(s, min_distance=min_distance, exclude_border=False)
    if len(pk) < 2:
        return None
    dy, dx = pk[:, 0] - _c, pk[:, 1] - _c
    d = np.hypot(dy, dx)
    keep = d > min_distance
    dy, dx, d = dy[keep], dx[keep], d[keep]
    if not len(d):
        return None
    o = np.argsort(d)[:n]
    return dy[o], dx[o], d[o]

def sac_orientation(sac, min_distance=2):
    """Grid orientation in degrees, modulo 60.

    Taken as the circular mean of the angles of the six SAC peaks nearest the
    centre. Angles are wrapped to [0, 60) because a hexagonal lattice is
    invariant under 60-degree rotation, then mapped onto the full circle
    (x6) so a circular mean is well defined, and mapped back."""
    pk = sac_peaks(sac, min_distance)
    if pk is None:
        return np.nan
    dy, dx, _ = pk
    ang = np.degrees(np.arctan2(dy, dx)) % 60.0
    a = np.radians(ang * 6.0)
    m = np.degrees(np.arctan2(np.sin(a).mean(), np.cos(a).mean())) % 360.0
    return float(m / 6.0)

def circ_spread_60(vals):
    """Circular SD (degrees) of orientations living on a 60-degree circle."""
    v = np.asarray(vals, float); v = v[np.isfinite(v)]
    if len(v) < 2:
        return np.nan
    a = np.radians(v * 6.0)
    R = np.hypot(np.sin(a).mean(), np.cos(a).mean())
    R = min(max(R, 1e-12), 1.0)
    return float(np.degrees(np.sqrt(-2 * np.log(R))) / 6.0)

print('SAC helpers defined (incl. orientation)')

SAC helpers defined


## Build the SAC matrix

All cells (not just grid cells) from the selected sessions — including
non-grid cells is the point: it lets us see whether the grid clusters are
actually grid-specific, or whether spatially-tuned non-grid cells land in the
same neighbourhoods.

In [3]:
def build_sacs(sel_sessions, force=False):
    cache = f'{SAC_CACHE}/sacs_{len(sel_sessions)}sessions_allcells_ori.npz'
    meta_c = cache.replace('.npz', '_meta.parquet')
    if os.path.exists(cache) and os.path.exists(meta_c) and not force:
        return np.load(cache)['X'], pd.read_parquet(meta_c)

    X, meta, t0 = [], [], time.time()
    for k, (mouse, day, session) in enumerate(sel_sessions, 1):
        beh_p, clu_p = session_files(mouse, day, session)
        if not (os.path.exists(beh_p) and os.path.exists(clu_p)):
            continue
        beh = nap.load_file(beh_p); clusters = nap.load_file(clu_p)
        ids, maps = compute_rate_maps(clusters, beh)
        sc = scores[(scores.mouse == mouse) & (scores.day == day) &
                    (scores.session == session)].set_index('cluster_id')
        for i, cid in enumerate(ids):
            if cid not in sc.index:
                continue
            sac = sac_fft(maps[i])
            X.append(sac_vector(sac))
            row = sc.loc[cid]
            meta.append(dict(mouse=mouse, day=day, session=session, cluster_id=cid,
                             cell_class=row['cell_class'], grid_score=row['grid_score'],
                             spatial_information=row['spatial_information'],
                             hd=row.get('hd_mean_vector_length', np.nan),
                             spacing_cm=sac_spacing(sac),
                             orientation_deg=sac_orientation(sac)))
        print(f'[{k}/{len(sel_sessions)}] M{mouse} D{day} {session} — '
              f'{len(X)} cells so far, {time.time()-t0:.0f}s', flush=True)
    X = np.asarray(X); meta = pd.DataFrame(meta)
    np.savez_compressed(cache, X=X); meta.to_parquet(meta_c)
    return X, meta

sel_sessions = [tuple(r) for r in sel[['mouse', 'day', 'session']].values] if len(sel) else []
if not sel_sessions:
    print('No sessions meet the grid-cell threshold yet — run the v2 sweep first.')
    X, meta = np.empty((0, 0)), pd.DataFrame()
else:
    X, meta = build_sacs(sel_sessions, force=True)
    print(f'\nSAC matrix: {X.shape}  ({meta.cell_class.value_counts().to_dict()})')

[1/1] M25 D24 OF1 — 308 cells so far, 2s



SAC matrix: (308, 1257)  ({'NGS': 164, 'NS': 94, 'GC': 50})


## UMAP embedding and clustering

DBSCAN eps is swept over the paper's 0.6-1.0 range; the chosen value is
reported with how many clusters and what fraction of cells it leaves as noise,
so the choice is visible rather than buried.

## Per-session grid modules from autocorrelogram structure

Everything here is **per session**. Modules are defined over simultaneously recorded
cells, so a cluster pooled across sessions would at best be a spacing band shared by
different animals, not a module.

Each session's curated cells (GC, NGS, NS together) are embedded by UMAP on their
spatial autocorrelograms and clustered with DBSCAN, eps set by the k-distance knee.

A cluster is called a **module** only if it is both grid-enriched and geometrically
coherent:

| criterion | threshold | why |
|---|---|---|
| grid cells in cluster | >= `MIN_GC_CLUSTER` | a module needs members |
| enrichment over session GC rate | >= `MIN_ENRICH` | grid-dominated, not incidental |
| spacing IQR (GC members) | <= `MAX_SPACING_IQR` | one scale, not a mixture |
| orientation circular SD (GC members) | <= `MAX_ORIENT_SD` | one lattice angle |

Enrichment alone is not enough: a cluster can be full of grid cells drawn from several
modules. The coherence criteria are what make it a module rather than a grid-cell bin.

**Grid cells outside every qualifying cluster are left unlabelled.** They remain grid
cells; they simply have no module assignment. Coverage — the fraction of grid cells that
receive a label — is reported, not maximised. A method that confidently labels half the
grid cells and abstains on the rest is more useful than one that forces every cell into
a module.

In [ ]:
# --- per-session modules: UMAP + knee-eps DBSCAN, enrichment + coherence gated ---
from sklearn.neighbors import NearestNeighbors

MIN_CELLS_SESSION = 60      # cells needed to embed
MIN_GC_SESSION    = 12      # grid cells needed to look for modules
DB_MIN_SESSION    = 3       # swept 3-6: 3 gives 24/28 sessions a module vs 17 at 5,
                            # with reproducibility still 1.000 (and on 9 testable days, not 6)
MIN_GC_CLUSTER    = 5       # grid cells for a cluster to be a candidate module
MIN_ENRICH        = 2.0     # x the session's own GC rate. Swept 1.5-2.5: this is a real
                            # threshold, not a convention -- at 1.75 cross-session
                            # reproducibility falls to 0.897, at 1.5 to 0.768.
MAX_SPACING_IQR   = 10.0    # cm
MAX_ORIENT_SD     = 10.0    # degrees, circular on the 60-degree circle


def knee_eps(emb, k=DB_MIN_SESSION):
    """DBSCAN eps at the k-distance knee: the point of the sorted k-NN distance
    curve furthest from its endpoint chord. No cluster-count preference, unlike a
    silhouette sweep (which collapsed to the smallest eps offered)."""
    d, _ = NearestNeighbors(n_neighbors=min(k, len(emb) - 1)).fit(emb).kneighbors(emb)
    kd = np.sort(d[:, -1]); x = np.arange(len(kd), dtype=float)
    x1, y1, x2, y2 = 0.0, kd[0], float(len(kd) - 1), kd[-1]
    num = np.abs((y2 - y1) * x - (x2 - x1) * kd + x2 * y1 - y2 * x1)
    return float(kd[np.argmax(num / np.hypot(y2 - y1, x2 - x1))])


per_session, clus_rows, sess_rows = {}, [], []
for (mo, dy, se), sub in meta.groupby(['mouse', 'day', 'session']):
    n_gc_sess = int((sub.cell_class == 'GC').sum())
    if len(sub) < MIN_CELLS_SESSION or n_gc_sess < MIN_GC_SESSION:
        continue
    emb = umap.UMAP(metric='manhattan', n_neighbors=int(min(15, max(3, len(sub) // 20))),
                    min_dist=0.05, init='spectral', n_components=2,
                    random_state=42).fit_transform(X[sub.index.values])
    eps = knee_eps(emb)
    lab = DBSCAN(eps=eps, min_samples=DB_MIN_SESSION).fit_predict(emb)

    s2 = sub.copy()
    s2['umap_x'], s2['umap_y'], s2['cluster'] = emb[:, 0], emb[:, 1], lab
    s2['module'] = np.nan                      # NaN = grid cell with no module label
    base = max((s2.cell_class == 'GC').mean(), 1e-9)

    mod_id = 0
    for cl in sorted(c for c in set(lab) if c >= 0):
        m = s2[s2.cluster == cl]
        g = m[m.cell_class == 'GC'].dropna(subset=['spacing_cm', 'orientation_deg'])
        enr = (m.cell_class == 'GC').mean() / base
        iqr = (g.spacing_cm.quantile(.75) - g.spacing_cm.quantile(.25)) if len(g) > 1 else np.nan
        osd = circ_spread_60(g.orientation_deg) if len(g) > 1 else np.nan
        is_mod = (len(g) >= MIN_GC_CLUSTER and enr >= MIN_ENRICH
                  and np.isfinite(iqr) and iqr <= MAX_SPACING_IQR
                  and np.isfinite(osd) and osd <= MAX_ORIENT_SD)
        if is_mod:
            s2.loc[g.index, 'module'] = mod_id
        clus_rows.append(dict(mouse=mo, day=dy, session=se, cluster=cl, n=len(m), n_gc=len(g),
                              pct_gc=round(100 * (m.cell_class == 'GC').mean(), 1),
                              enrichment=round(enr, 2),
                              spacing=round(g.spacing_cm.median(), 1) if len(g) else np.nan,
                              spacing_iqr=round(iqr, 1) if np.isfinite(iqr) else np.nan,
                              orient=round(g.orientation_deg.median(), 1) if len(g) else np.nan,
                              orient_sd=round(osd, 1) if np.isfinite(osd) else np.nan,
                              is_module=is_mod, module=mod_id if is_mod else np.nan))
        mod_id += int(is_mod)

    per_session[(mo, dy, se)] = s2
    n_lab = int(s2.module.notna().sum())
    sess_rows.append(dict(mouse=mo, day=dy, session=se, n_cells=len(s2), n_gc=n_gc_sess,
                          eps=round(eps, 3), n_clusters=len(set(lab)) - (1 if -1 in lab else 0),
                          n_modules=mod_id, gc_labelled=n_lab,
                          coverage_pct=round(100 * n_lab / max(n_gc_sess, 1), 1)))

sess_df = pd.DataFrame(sess_rows); clus_df = pd.DataFrame(clus_rows)
print(f'{len(sess_df)} sessions | {int(sess_df.n_modules.sum())} modules found | '
      f'median coverage {sess_df.coverage_pct.median():.0f}% of grid cells labelled')
print(sess_df.to_string(index=False))


In [ ]:
# --- what the modules look like, and what was left unlabelled ---
print('=== clusters called modules ===')
mods = clus_df[clus_df.is_module]
print(mods.drop(columns=['is_module']).to_string(index=False) if len(mods) else '  none')

print('\n=== grid-enriched clusters REJECTED for incoherence ===')
rej = clus_df[(~clus_df.is_module) & (clus_df.n_gc >= MIN_GC_CLUSTER)
              & (clus_df.enrichment >= MIN_ENRICH)]
if len(rej):
    r = rej.copy()
    r['why'] = np.where(r.spacing_iqr > MAX_SPACING_IQR,
                        np.where(r.orient_sd > MAX_ORIENT_SD, 'spacing + orientation', 'spacing spread'),
                        'orientation spread')
    print(r[['mouse','day','session','cluster','n_gc','enrichment','spacing_iqr','orient_sd','why']]
          .to_string(index=False))
    print(f'\n  {len(rej)} clusters were grid-enriched but not coherent -- these are the ones a '
          f'purely enrichment-based rule would have wrongly called modules.')
else:
    print('  none')

print('\n=== coverage ===')
tot_gc = int(sess_df.n_gc.sum()); tot_lab = int(sess_df.gc_labelled.sum())
print(f'  {tot_lab}/{tot_gc} grid cells ({100*tot_lab/tot_gc:.1f}%) received a module label')
print(f'  {tot_gc-tot_lab} grid cells are unlabelled -- grid cells with no module assignment')
print(f'  sessions yielding >=1 module: {int((sess_df.n_modules>0).sum())}/{len(sess_df)}')
print(f'  sessions yielding >=2 modules: {int((sess_df.n_modules>=2).sum())}/{len(sess_df)}')


In [ ]:
# --- reproducibility, on labelled cells only ---
# Module IDs are arbitrary per session, so agreement is measured pairwise and
# label-invariantly: for cells labelled in BOTH sessions of a day, does each pair
# agree on same-module vs different-module? This scores precision on the cells the
# method committed to, which is the relevant question when abstention is allowed.
rows = []
for (mo, dy), grp in pd.DataFrame(
        [dict(mouse=k[0], day=k[1], session=k[2]) for k in per_session]).groupby(['mouse', 'day']):
    if set(grp.session) != {'OF1', 'OF2'}:
        continue
    a = per_session[(mo, dy, 'OF1')].set_index('cluster_id')
    b = per_session[(mo, dy, 'OF2')].set_index('cluster_id')
    common = [c for c in a.index.intersection(b.index)
              if pd.notna(a.loc[c, 'module']) and pd.notna(b.loc[c, 'module'])]
    if len(common) < 4:
        rows.append(dict(mouse=mo, day=dy, n_labelled_both=len(common), pair_agree=np.nan)); continue
    ma = a.loc[common, 'module'].values; mb = b.loc[common, 'module'].values
    same_a = ma[:, None] == ma[None, :]; same_b = mb[:, None] == mb[None, :]
    iu = np.triu_indices(len(common), 1)
    rows.append(dict(mouse=mo, day=dy, n_labelled_both=len(common),
                     pair_agree=round((same_a[iu] == same_b[iu]).mean(), 3)))
rep = pd.DataFrame(rows)
print('reproducibility of module labels across the two open-field sessions:')
print(rep.to_string(index=False))
v = rep.dropna(subset=['pair_agree'])
if len(v):
    print(f'\n  median pairwise agreement {v.pair_agree.median():.3f} over {len(v)} days '
          f'(chance depends on module count; 0.5 for two equal modules)')
